In [7]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(scutilsR)
library(tidyverse)
library(celda)
library(sceasy)
library(Nebulosa)
library(glue)

In [ ]:
seu = qs::qread("/home/data/tanglei/project/prostate_altas/output/04/QC_fin.qs")

In [ ]:
## miloR + Augur: type 一对多 (Target vs Other) × H2/H3
library(miloR)
library(SingleCellExperiment)
library(scater)
library(scran)
library(dplyr)
library(ggplot2)
library(Augur)
library(ggrepel)

out_dir <- "/home/data/tanglei/project/prostate_altas/output/04/miloR+AuguR"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

types <- c("5aTreat_BPH", "Cribriform_PC", "CRPC_A", "Naive_BPH", "Naive_PC", "NEPC", "Normal")
ct_cols <- c("celltype_manual_H2", "celltype_manual_H3")

## 子集到 7 类
seu_sub <- subset(seu, subset = type %in% types)
message("Cells after subset: ", ncol(seu_sub))
print(table(seu_sub$type))

## 一次性构建 milo 图结构（与 celltype / target 无关）
rpca.embeddings <- Embeddings(seu_sub, reduction = "rpca")
umap.embeddings <- Embeddings(seu_sub, reduction = "umap.rpca")

sce <- SingleCellExperiment(
  assays = list(counts = GetAssayData(seu_sub, assay = "RNA", layer = "counts")),
  reducedDims = SimpleList(RPCA = rpca.embeddings, UMAP = umap.embeddings)
)

milo_base <- Milo(sce)
milo_base <- buildGraph(milo_base, k = 30, d = 30, reduced.dim = "RPCA")
milo_base <- makeNhoods(milo_base, prop = 0.1, k = 30, d = 30, refined = TRUE, reduced_dims = "RPCA")
milo_base <- calcNhoodDistance(milo_base, d = 30, reduced.dim = "RPCA")
message("milo graph ready")

## 双层循环：celltype 层级 × target
for (ct_col in ct_cols) {
  ct_level <- ifelse(grepl("H2$", ct_col), "H2", "H3")
  message("==== celltype: ", ct_col, " (", ct_level, ") ====")

  for (target in types) {
    message("-- target: ", target, " vs Other | ", ct_col)

    tryCatch({
      ## 二分类标签
      group_vec <- ifelse(seu_sub$type == target, "Target", "Other")
      group_vec <- factor(group_vec, levels = c("Other", "Target"))
      sample_vec <- paste0(seu_sub$orig.ident, ".", as.character(group_vec))
      celltype_vec <- as.character(seu_sub[[ct_col, drop = TRUE]])

      meta.data <- data.frame(
        sample = sample_vec,
        group = group_vec,
        celltype = celltype_vec,
        row.names = colnames(seu_sub),
        stringsAsFactors = FALSE
      )

      milo <- milo_base
      colData(milo) <- DataFrame(meta.data)
      milo <- countCells(milo, meta.data = as.data.frame(colData(milo)), sample = "sample")

      milo_design <- as.data.frame(colData(milo)) %>%
        dplyr::select(sample, group) %>%
        distinct()
      milo_design$group <- factor(milo_design$group, levels = c("Other", "Target"))
      rownames(milo_design) <- milo_design$sample

      da_results <- testNhoods(milo, design = ~ group, design.df = milo_design, reduced.dim = "RPCA")
      da_results <- annotateNhoods(milo, da_results, coldata_col = "celltype")
      da_results$celltype <- ifelse(da_results$celltype_fraction < 0.7, "Mixed", da_results$celltype)

      plot_title <- paste0(target, " vs Other | ", ct_col)

      p_milo <- plotDAbeeswarm(da_results, group.by = "celltype", alpha = 1) +
        scale_color_gradient2(midpoint = 0, low = "blue", mid = "white", high = "red", space = "Lab") +
        geom_hline(yintercept = 0, linetype = "dashed") +
        ggtitle(plot_title)
      ggsave(
        file.path(out_dir, paste0("miloR_", ct_level, "_", target, ".pdf")),
        p_milo, width = 8, height = 6
      )

      ## Augur
      seu_sub$augur_label <- as.character(group_vec)
      augur <- calculate_auc(
        seu_sub,
        cell_type_col = ct_col,
        label_col = "augur_label",
        n_threads = 1
      )

      data.plot <- subset(augur$results, metric == "roc_auc")
      data.plot$cell_type <- factor(data.plot$cell_type, levels = rev(augur$AUC$cell_type))
      p_augur_violin <- ggplot(data.plot, aes(cell_type, estimate, color = cell_type)) +
        geom_violin() +
        geom_jitter(size = .3) +
        coord_flip() +
        theme_bw() +
        theme(legend.position = "none") +
        ggtitle(plot_title)
      ggsave(
        file.path(out_dir, paste0("Augur_violin_", ct_level, "_", target, ".pdf")),
        p_augur_violin, width = 7, height = 6
      )

      data.bar <- augur$AUC
      data.bar$cell_type <- factor(data.bar$cell_type, levels = rev(augur$AUC$cell_type))
      p_augur_bar <- ggplot(data.bar, aes(cell_type, auc, fill = cell_type)) +
        geom_col() +
        coord_flip() +
        theme_bw() +
        theme(legend.position = "none") +
        ggsci::scale_fill_d3("category20") +
        ggtitle(plot_title)
      ggsave(
        file.path(out_dir, paste0("Augur_bar_", ct_level, "_", target, ".pdf")),
        p_augur_bar, width = 7, height = 6
      )

      ## Combined (errorbar)
      data1 <- da_results %>%
        group_by(celltype) %>%
        summarise(logFC = mean(logFC), xerr = sd(logFC), .groups = "drop")
      colnames(data1) <- c("cell_type", "logFC", "xerr")

      data2 <- subset(augur$results, metric == "roc_auc") %>%
        group_by(cell_type) %>%
        summarise(auc = mean(estimate), yerr = sd(estimate), .groups = "drop")

      data_plot <- inner_join(data1, data2, by = "cell_type")
      ## Mixed 无 Augur 条目时自然被 drop；NA sd 置 0
      data_plot$xerr[is.na(data_plot$xerr)] <- 0
      data_plot$yerr[is.na(data_plot$yerr)] <- 0

      p_combined <- ggplot(data_plot, aes(x = logFC, y = auc, color = cell_type, label = cell_type)) +
        geom_point(size = 4) +
        geom_errorbar(aes(ymin = auc - yerr, ymax = auc + yerr), width = 0.1, alpha = 0.3) +
        geom_errorbarh(aes(xmin = logFC - xerr, xmax = logFC + xerr), height = 0.1, alpha = 0.3) +
        geom_hline(yintercept = 0.5, linetype = "dashed", color = "black") +
        geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
        ggrepel::geom_text_repel(show.legend = FALSE, color = "black") +
        theme_bw(base_size = 15) +
        labs(x = "logFC by MiloR", y = "AUC score by Augur", title = plot_title) +
        theme(
          legend.position = "none",
          panel.grid.major = element_blank(),
          panel.grid.minor = element_blank()
        )
      ggsave(
        file.path(out_dir, paste0("Combined_", ct_level, "_", target, ".pdf")),
        p_combined, width = 8, height = 7
      )

      message("  done: ", target, " / ", ct_level)
    }, error = function(e) {
      message("  FAILED: ", target, " / ", ct_level, " -> ", conditionMessage(e))
    })
  }
}

message("All finished. PDFs in: ", out_dir)